<a href="https://colab.research.google.com/github/Patrick190508/whaleshark-mafia-island/blob/main/notebooks/OBIS_Dataset_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# From Raw OBIS to OBIS + Copernicus Dataset


## Step 1 - upload OBIS dataset in the nootebook

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

db = files.upload()
db = pd.read_csv('Occurrence.tsv', sep="\t", low_memory=False)

Saving Occurrence.tsv to Occurrence.tsv


## Step 2 - Cleaning the records that are not usable for the research

### 1) Filtering and correcting by Data values

Completing the 4 columns: Date, Year, Month, Day using date_mid (mid values from date_start and date_end)

In [ ]:
db['date'] = pd.to_datetime(db['date_mid'], unit='ms')
db['year'] = db['date'].dt.year
db['month'] = db['date'].dt.month
db['day'] = db['date'].dt.day
db[['eventDate', 'date', 'year', 'month', 'day']].head()

,eventDate,date,year,month,day
0,1993-05-30T18:55:42,1993-05-30,"1,993.0",5.0,30.0
1,2008-01-27,2008-01-27,"2,008.0",1.0,27.0
2,2012-01-07,2012-01-07,"2,012.0",1.0,7.0
3,2011-03-19,2011-03-19,"2,011.0",3.0,19.0
4,2014-11-09,2014-11-09,"2,014.0",11.0,9.0


Eliminating records with:

*   No Date
*   Too old (befor september 1981)
*   With at least month precision





In [ ]:
print("start:", len(db))

db = db[db['date'].notna()]
print("with data:", len(db))

db = db[db['date'] >= '1981-09-01']
print("from september 1981:", len(db))

# Each record has the values date_start & date_end (in millisecond) this range is used not 0 if there is a range of uncertainity for the date of the encounter
day_interval = (db['date_end'] - db['date_start']) / (1000*60*60*24)
# With the variable day_interval we exclude any records which day is not sure (from initial analysis the 75th percentile of day_interval was 0 so few records excluded)
db = db[day_interval <= 1]
print("precise date within day", len(db))

start: 17224
with data: 16976
from september 1981: 16940
precise date within day 16477


**Before Date filtering: 17224 Records**

**After Date filtering: 16477 Records**



---




### 2) Filtering by types of observation (human recordings/telemetry)

As asked by Samuel we will use only Human recordings

In [ ]:
print("All records: ", len(db))
db['basisOfRecord'] = db['basisOfRecord'].str.lower()
db = db[db['basisOfRecord'] == 'humanobservation']
print("Only human sightings:", len(db))

All records:  16477
Only human sightings: 10505


**Before Date types filtering: 16477 Records**

**After Date types filtering: 10505 Records**



---



### 3) Eliminating bathymetry errors (records on land)

Since eliminating all the records with negative bathymetry would not consider the error due to the precision of the bathymetry measurement we will insert a thresold using the shoredistance of the sightings and flag the negative-bathymetry records.

In [ ]:
print("All records: ", len(db))
on_land = db[db['bathymetry'] <= 0]
on_land = on_land[((on_land['shoredistance']) <= 500)&((on_land['shoredistance']) >= 0)]
on_land = on_land[(on_land['coordinateUncertaintyInMeters'].isna())|(on_land['coordinateUncertaintyInMeters'] <= 300)]
on_sea = db[db['bathymetry'] > 0]
db = pd.concat([on_land, on_sea])
db = db.copy()
db['is_on_land'] = db['bathymetry'] <= 0
print("Only on sea or near coast records: ", len(db))
print("On land sightings: ", len(db[db['is_on_land'] == True]))

All records:  10505
Only on sea or near coast records:  9879
On land sightings:  150


***Run this code if you want to exclude the near coast records***

In [ ]:
print("All records: ", len(db))
#db = db[db['bathymetry'] >= 0]
print("Only on sea records: ", len(db))

All records:  9879
Only on sea records:  9879


**Before Date types filtering: 10505 Records**

**After Date types filtering: 9879 Records (150 flagged with is_on_land = True)**

### 4) Analyzing uncertainity in records and marking uncertainity with flags

To train our model we need to establish if a record is precise or not

In [ ]:
pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_columns', None)
db['coordinateUncertaintyInMeters'].describe()

,coordinateUncertaintyInMeters
count,"1,597.0"
mean,"44,263.4"
std,"168,980.7"
min,0.1
25%,500.0
50%,"30,357.0"
75%,"31,254.0"
max,"2,931,896.0"


**1597/9879 records are obscured**




I will put a thresold eliminate the records with an uncertainity over 40km

In [ ]:
print("All records:", len(db))
inc = db['coordinateUncertaintyInMeters']
db['uncertainity'] = inc >= 20000
db = db[~(inc > 40000)]
print("After removing uncertainty > 40 km:", len(db))
print(db['uncertainity'].value_counts())

All records: 9879
After removing uncertainty > 40 km: 9754
uncertainity
False    8823
True      931
Name: count, dtype: int64


**Before obscuring filtering: 9879 Records**


**After obscuring filtering: 9754 Records**


### 5) Removing Duplicated records

Let's check how many duplicated records we have by analyzing the dataset_id, decimalLatitude, decimalLongitude, Date, and catalogNumber (id assigned by the single dataset)

In [ ]:
keys = ['dataset_id', 'decimalLatitude', 'decimalLongitude', 'date', 'catalogNumber']
print("Duplicated Records", db.duplicated(subset=keys).sum())

Duplicated Records 9


**9 duplicated records**

In [ ]:
db = db.drop_duplicates(subset=keys)
print("Cleaned Dataset", len(db))

Cleaned Dataset 9745


**Before eliminating duplicates: 9754 Records**


**After eliminating duplicates: 9745 Records**


### 6) Removing dead encounter (using the vitality parameters)

In [ ]:
print("dead sharks:", (db['vitality'] == 'dead').sum())
db = db[db['vitality'] != 'dead']
print("Records after remooving dead sharks", len(db))

dead sharks: 14
Records after remooving dead sharks 9731


## Step 3 eliminating the empty or not needed columns

### 1) Eliminating Empty columns

In [ ]:
print("All the columns:", db.shape[1])
db = db.dropna(axis=1, how="all")
print("Only columns with at least 1 value", db.shape[1])

All the columns: 285
Only columns with at least 1 value 128


**Before Eliminating empty columns: 285 Columns**

**After Eliminating empty columns: 128 Columns**

### 2) Eliminating Constant values columns

Columns that only show one value don't give any information about the records

In [ ]:
print("Before Eliminating constant columns:", db.shape[1])
costant_columns = [c for c in db.columns if db[c].nunique(dropna=False) == 1]
db = db.drop(costant_columns, axis=1)
print("After Eliminating constant columns:", db.shape[1])

Before Eliminating constant columns: 128
After Eliminating constant columns: 92


**Before Eliminating constant columns: 128 Columns**

**After Eliminating constant columns: 92 Columns**


### 3) Keeping only useful columns for the Data analysis

In [ ]:
keep = ["dataset_id", "id", "occurrenceID", "catalogNumber", "institutionCode", "datasetName", "decimalLatitude", "decimalLongitude", "coordinateUncertaintyInMeters", "date", "date_start", "date_end", "year", "month", "day", "eventTime", "bathymetry", "shoredistance", "sst", "sss", "flags", "is_on_land", "uncertainity"]
db = db[keep]
print(db.shape)


NameError: name 'db' is not defined

Final Dataset Shape after first cleaning process: **9731 rows**, **23 columns**

## Step 4 - Copernicus data connection

### 1) Installing dependencies and logging in the copernicus account

*Username: psilingardi*

*Password: Patrick1908@*

In [ ]:
!pip install -q copernicusmarine
import copernicusmarine
copernicusmarine.login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.5/130.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.7 MB/s eta 0:00:00


INFO - 2026-09-13T17:11:42Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register
INFO:copernicusmarine:Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username: psilingardi
Copernicus Marine password: ··········


INFO - 2026-09-13T17:11:57Z - Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.
INFO:copernicusmarine:Credentials file stored in /root/.copernicusmarine/.copernicusmarine-credentials.


True

In [ ]:
sst_ds = copernicusmarine.open_dataset(
    dataset_id="METOFFICE-GLO-SST-L4-REP-OBS-SST",
    variables=["analysed_sst"]
)
print(sst_ds)

INFO - 2026-09-13T17:12:43Z - Selected dataset version: "202003"
INFO:copernicusmarine:Selected dataset version: "202003"
INFO - 2026-09-13T17:12:43Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


<xarray.Dataset> Size: 3TB
Dimensions:       (time: 16253, latitude: 3600, longitude: 7200)
Coordinates:
  * time          (time) datetime64[ns] 130kB 1981-10-01 ... 2026-03-31
  * latitude      (latitude) float32 14kB -89.97 -89.93 -89.88 ... 89.93 89.97
  * longitude     (longitude) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables:
    analysed_sst  (time, latitude, longitude) float64 3TB dask.array<chunksize=(50, 1024, 2048), meta=np.ndarray>
Attributes: (12/48)
    Conventions:                CF-1.4, ACDD-1.3
    Metadata_Conventions:       Unidata Observation Dataset v1.0
    acknowledgment:             Please acknowledge the use of these data with...
    cdm_data_type:              grid
    comment:                    WARNING Some applications are unable to prope...
    creator_email:              servicedesk.cmems@mercator-ocean.eu
    ...                         ...
    time_coverage_end:          19811002T000000Z
    time_coverage_start:        19811001T000000Z

In [ ]:
import numpy as np, xarray as xr, os

# --- preparazione: colonna vuota, o ripresa da un salvataggio precedente
if os.path.exists("sst_progress.csv"):
    prog = pd.read_csv("sst_progress.csv")
    db['sst_cmems'] = db['id'].map(prog.set_index('id')['sst_cmems'])
    print("ripreso:", db['sst_cmems'].notna().sum(), "già fatti")
else:
    db['sst_cmems'] = np.nan

# --- solo i record ancora da fare, e solo nel periodo coperto dal dataset REP (fino al 2022)
da_fare = db[db['sst_cmems'].isna() & (db['date'] <= '2022-05-31')]
print("da fare:", len(da_fare))

BLOCCO = 300
for inizio in range(0, len(da_fare), BLOCCO):
    b = da_fare.iloc[inizio:inizio+BLOCCO]
    pts = sst_ds['analysed_sst'].sel(
        time=xr.DataArray(pd.to_datetime(b['date']).values, dims="p"),
        latitude=xr.DataArray(b['decimalLatitude'].values, dims="p"),
        longitude=xr.DataArray(b['decimalLongitude'].values, dims="p"),
        method="nearest"
    ).values
    db.loc[b.index, 'sst_cmems'] = pts - 273.15
    db[['id', 'sst_cmems']].to_csv("sst_progress.csv", index=False)
    print(f"{inizio+len(b)}/{len(da_fare)}", end="  ")

-24.8973 153.2721 2003-09-09 00:00:00
SST: 21.899993406981253 °C
